In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch.nn as nn

# PATHS

In [ ]:
CKPT_PATH = "/Users/YGT/ist-airport-decision-support-system/src/modeling/atscc_checkpoints_v10_reg/atscc_best.pt"
TRAJ_PATH = "/Users/YGT/ist-airport-decision-support-system/data/gold/trajectories/trajectory_gold.parquet"
EMB_PATH = "/Users/YGT/ist-airport-decision-support-system/data/gold/atscc_embeddings_final_all_fixed.parquet"
FULL_EMB_PATH = "/Users/YGT/ist-airport-decision-support-system/data/gold/atscc_embeddings_final_all_fixed.parquet"
N_FLIGHTS_CHECK = 128
MAX_POINTS = 256

# DEVICE

In [5]:
def get_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"

device = get_device()

print("Device:", device)

Device: mps


# 1.) Checkpoint embedding quality

# MODEL CLASS (ATSCC)

In [6]:
class TransformerLayer(nn.Module):

    def __init__(self, d_model, n_head, d_ff, dropout):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_head,
            batch_first=True
        )

        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x, mask, kpm):

        attn_out,_ = self.attn(
            self.norm1(x),
            self.norm1(x),
            self.norm1(x),
            attn_mask=mask,
            key_padding_mask=kpm,
            need_weights=False
        )

        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x


class ATSCCEncoder(nn.Module):

    def __init__(self, cfg):

        super().__init__()

        self.in_proj = nn.Linear(len(cfg["feature_cols"]), cfg["d_model"])
        self.in_ln   = nn.LayerNorm(cfg["d_model"])

        self.layers = nn.ModuleList([
            TransformerLayer(
                cfg["d_model"],
                cfg["n_head"],
                cfg["d_ff"],
                cfg["dropout"]
            )
            for _ in range(cfg["n_layer"])
        ])

        self.final_ln = nn.LayerNorm(cfg["d_model"])
        self.out_proj = nn.Linear(cfg["d_model"], cfg["d_out"])
        self.out_ln   = nn.LayerNorm(cfg["d_out"])


    def forward(self, x, kpm):

        x = torch.nan_to_num(x)

        h = F.normalize(self.in_ln(self.in_proj(x)), p=2, dim=-1)

        T = h.shape[1]

        mask = torch.triu(
            torch.ones(T,T,device=h.device,dtype=torch.bool),
            diagonal=1
        )

        for layer in self.layers:
            h = layer(h, mask, kpm)

        h = self.final_ln(h)

        z = F.normalize(self.out_ln(self.out_proj(h)), p=2, dim=-1)

        return z



# MODEL LOAD

In [7]:
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

cfg = ckpt["config"]

model = ATSCCEncoder(cfg).to(device)

model.load_state_dict(ckpt["model_state_dict"])

model.eval()

ATSCCEncoder(
  (in_proj): Linear(in_features=11, out_features=192, bias=True)
  (in_ln): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
  (layers): ModuleList(
    (0-3): 4 x TransformerLayer(
      (norm1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=192, out_features=192, bias=True)
      )
      (ff): Sequential(
        (0): Linear(in_features=192, out_features=768, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.25, inplace=False)
        (3): Linear(in_features=768, out_features=192, bias=True)
      )
    )
  )
  (final_ln): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
  (out_proj): Linear(in_features=192, out_features=256, bias=True)
  (out_ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
)

# RAW TRAJECTORY LOAD

In [8]:
traj = pd.read_parquet(TRAJ_PATH)

traj["point_timestamp"] = pd.to_datetime(traj["point_timestamp"], utc=True)

traj = traj.sort_values(["flight_id","point_timestamp"])

flight_groups = {
    fid:g.reset_index(drop=True)
    for fid,g in traj.groupby("flight_id")
}

print("Flights:", len(flight_groups))

Flights: 566791


# SAMPLE FLIGHTS

In [9]:
rng = np.random.default_rng(42)

sample_ids = rng.choice(
    list(flight_groups.keys()),
    size=min(N_FLIGHTS_CHECK, len(flight_groups)),
    replace=False
)

print("Sample flights:", len(sample_ids))

Sample flights: 128


# TOKEN EMBEDDINGS

In [13]:
all_z = []
all_seg = []
all_fid = []

with torch.no_grad():

    for fid in sample_ids:

        g = flight_groups[fid]

        cols = list(cfg["feature_cols"]) + ["segment_id"]

        if not all(c in g.columns for c in cols):
            continue

        g = g.dropna(subset=cols)

        if len(g) < 8:
            continue

        if len(g) > MAX_POINTS:
            g = g.iloc[-MAX_POINTS:]

        x = torch.tensor(
            g[list(cfg["feature_cols"])].to_numpy(dtype=np.float32),
            device=device
        ).unsqueeze(0)

        kpm = torch.zeros((1,len(g)),dtype=torch.bool,device=device)

        z = model(x,kpm)[0].cpu()

        seg = torch.tensor(g["segment_id"].to_numpy())

        fid_vec = torch.full((len(g),), hash(fid)%1000000)

        all_z.append(z)
        all_seg.append(seg)
        all_fid.append(fid_vec)

z = torch.cat(all_z)
seg = torch.cat(all_seg)
fid = torch.cat(all_fid)

print("Token count:", len(z))

Token count: 6123


# NORM CHECK

In [14]:
norms = torch.norm(z,dim=1).numpy()

print("\nNorm check")
print("mean:",norms.mean())
print("std :",norms.std())


Norm check
mean: 1.0
std : 4.5550873e-08


# COSINE CHECK

In [15]:
z = F.normalize(z,p=2,dim=1)

N = len(z)

same = []
diff = []

for _ in range(20000):

    i = np.random.randint(0,N)
    j = np.random.randint(0,N)

    if i==j:
        continue

    sim = float((z[i]*z[j]).sum())

    if fid[i]==fid[j] and seg[i]==seg[j]:
        same.append(sim)
    else:
        diff.append(sim)

same = np.array(same)
diff = np.array(diff)

print("\nCosine check")

print("same segment:",same.mean())
print("diff segment:",diff.mean())

gap = same.mean()-diff.mean()

print("gap:",gap)


Cosine check
same segment: 0.751336644773614
diff segment: 0.10056251037713118
gap: 0.6507741343964828


# Analysis

In [16]:
print("\nQuick verdict")

if abs(norms.mean()-1)<0.05 and gap>0.15:
    print("GOOD embeddings")
elif gap>0.08:
    print("OK embeddings")
else:
    print("WEAK embeddings")


Quick verdict
GOOD embeddings


# 2.) Exported embedding file sanity check

In [18]:
EMB_PATH = "/Users/YGT/ist-airport-decision-support-system/data/gold/atscc_embeddings_final_all_fixed.parquet"

emb = pd.read_parquet(EMB_PATH)

emb_cols = [c for c in emb.columns if c.startswith("atscc_emb_")]
X = emb[emb_cols].to_numpy(dtype=np.float32)

norms = np.linalg.norm(X, axis=1)

print("shape:", emb.shape)
print("emb_dim:", len(emb_cols))
print("norm mean:", norms.mean())
print("norm std:", norms.std())
print("norm min/max:", norms.min(), norms.max())

print("\ncolumns check:")
print(emb[["flight_id", "hex_icao", "traj_start", "traj_end", "traj_len_raw", "traj_len_used", "flight_type"]].head())

shape: (566791, 263)
emb_dim: 256
norm mean: 1.0
norm std: 1.0858924e-07
norm min/max: 0.9999994 1.0000005

columns check:
  flight_id hex_icao                traj_start                  traj_end  \
0  000001_0   000001 2025-03-10 09:07:51+00:00 2025-03-10 09:12:02+00:00   
1  000001_1   000001 2025-03-10 13:40:11+00:00 2025-03-10 13:55:53+00:00   
2  000001_4   000001 2025-03-14 11:05:42+00:00 2025-03-14 11:27:40+00:00   
3  000016_0   000016 2025-10-08 04:31:55+00:00 2025-10-08 04:33:28+00:00   
4  00006a_0   00006A 2025-03-10 15:00:37+00:00 2025-03-10 15:02:49+00:00   

   traj_len_raw  traj_len_used flight_type  
0            18             18     unknown  
1            33             33     arrival  
2            49             49   departure  
3            19             19     arrival  
4            27             27     unknown  


In [19]:
FULL_EMB_PATH = "/Users/YGT/ist-airport-decision-support-system/data/gold/atscc_embeddings_final_all_fixed.parquet"

emb = pd.read_parquet(FULL_EMB_PATH)

emb_cols = [c for c in emb.columns if c.startswith("atscc_emb_")]
X = emb[emb_cols].to_numpy(dtype=np.float32)

norms = np.linalg.norm(X, axis=1)

print("shape:", emb.shape)
print("emb_dim:", len(emb_cols))
print("norm mean:", norms.mean())
print("norm std:", norms.std())
print("norm min/max:", norms.min(), norms.max())

print("\ncolumns check:")
print(emb[["flight_id", "hex_icao", "traj_start", "traj_end", "traj_len_raw", "traj_len_used", "flight_type"]].head())

shape: (566791, 263)
emb_dim: 256
norm mean: 1.0
norm std: 1.0858924e-07
norm min/max: 0.9999994 1.0000005

columns check:
  flight_id hex_icao                traj_start                  traj_end  \
0  000001_0   000001 2025-03-10 09:07:51+00:00 2025-03-10 09:12:02+00:00   
1  000001_1   000001 2025-03-10 13:40:11+00:00 2025-03-10 13:55:53+00:00   
2  000001_4   000001 2025-03-14 11:05:42+00:00 2025-03-14 11:27:40+00:00   
3  000016_0   000016 2025-10-08 04:31:55+00:00 2025-10-08 04:33:28+00:00   
4  00006a_0   00006A 2025-03-10 15:00:37+00:00 2025-03-10 15:02:49+00:00   

   traj_len_raw  traj_len_used flight_type  
0            18             18     unknown  
1            33             33     arrival  
2            49             49   departure  
3            19             19     arrival  
4            27             27     unknown  


# 3.) FULL EMBEDDING CONTROL

In [21]:
import pandas as pd
import numpy as np

# =========================================================
# PATHS
# =========================================================

TRAJ_PATH = "/Users/YGT/ist-airport-decision-support-system/data/gold/trajectories/trajectory_gold.parquet"

FULL_EMB_PATH = "/Users/YGT/ist-airport-decision-support-system/data/gold/atscc_embeddings_final_all_fixed.parquet"

# =========================================================
# LOAD DATA
# =========================================================

print("Loading embedding table...")
emb = pd.read_parquet(FULL_EMB_PATH)

print("Loading raw trajectories...")
traj = pd.read_parquet(TRAJ_PATH)

traj["point_timestamp"] = pd.to_datetime(traj["point_timestamp"], utc=True)

# =========================================================
# COMPUTE RAW BOUNDS
# =========================================================

print("Computing raw trajectory bounds...")

raw_bounds = (
    traj
    .groupby("flight_id")
    .agg(
        raw_start=("point_timestamp", "min"),
        raw_end=("point_timestamp", "max"),
        raw_len=("point_timestamp", "count"),
    )
    .reset_index()
)

# =========================================================
# MERGE
# =========================================================

print("Merging tables...")

df = emb.merge(raw_bounds, on="flight_id", how="left")

# =========================================================
# CHECK MISMATCH
# =========================================================

start_diff = (df["traj_start"] != df["raw_start"])
end_diff   = (df["traj_end"]   != df["raw_end"])

print("\n==============================")
print("BOUND CHECK")
print("==============================")

print("traj_start mismatch:", start_diff.sum())
print("traj_end   mismatch:", end_diff.sum())

# =========================================================
# TIME DIFFERENCE ANALYSIS
# =========================================================

df["start_delta_s"] = (
    df["traj_start"] - df["raw_start"]
).dt.total_seconds()

df["end_delta_s"] = (
    df["traj_end"] - df["raw_end"]
).dt.total_seconds()

print("\nStart delta statistics (seconds)")
print(df["start_delta_s"].describe())

print("\nEnd delta statistics (seconds)")
print(df["end_delta_s"].describe())

# =========================================================
# SHOW BAD EXAMPLES
# =========================================================

bad = df[(start_diff) | (end_diff)]

print("\n==============================")
print("BAD EXAMPLES")
print("==============================")

print(bad[[
    "flight_id",
    "traj_start","raw_start",
    "traj_end","raw_end"
]].head(10))

print("\nTotal mismatches:", len(bad))

# =========================================================
# QUICK VERDICT
# =========================================================

if len(bad) == 0:
    print("\nMükemmel: Metadata raw trajectory ile %100 uyumlu")
elif len(bad) < 100:
    print("\n Küçük sayıda mismatch var, incelemek gerekebilir")
else:
    print("\n Çok fazla mismatch var, export script gözden geçirilmeli")

Loading embedding table...
Loading raw trajectories...
Computing raw trajectory bounds...
Merging tables...

BOUND CHECK
traj_start mismatch: 0
traj_end   mismatch: 0

Start delta statistics (seconds)
count    566791.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: start_delta_s, dtype: float64

End delta statistics (seconds)
count    566791.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: end_delta_s, dtype: float64

BAD EXAMPLES
Empty DataFrame
Columns: [flight_id, traj_start, raw_start, traj_end, raw_end]
Index: []

Total mismatches: 0

Mükemmel: Metadata raw trajectory ile %100 uyumlu
